# SkyLine Airways RAG Chatbot — Stage 5: RAGAS Evaluation

**Goal:** Measure how well our RAG pipeline actually performs using four RAGAS metrics:

| Metric | What it measures | Needs ground truth? |
|---|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved chunks? (hallucination detector) | No |
| **Answer Relevancy** | Does the answer actually address the question? | No |
| **Context Precision** | Are the retrieved chunks relevant to the question? | Yes |
| **Context Recall** | Did we retrieve all the information needed to answer? | Yes |

All scores are 0–1. Higher is better.

**Test set:** 10 hand-crafted questions with ground-truth answers, covering all 10 policy documents.

In [1]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "ragas>=0.2.0,<0.3.0",
    "langchain==0.3.7",
    "langchain-core>=0.3.0,<0.4.0",
    "langchain-community==0.3.7",
    "langchain-openai==0.2.8",
    "sentence-transformers>=3.0.0",
    "datasets",
    "nest_asyncio",
    "httpx<0.28",
])

import importlib.metadata
print("ragas  :", importlib.metadata.version("ragas"))
print("langchain:", importlib.metadata.version("langchain"))

ragas  : 0.2.15
langchain: 0.3.7


In [2]:
import re, sys
from pathlib import Path
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

ROOT = Path.cwd().parent
DOCS_DIR = ROOT / "docs"
INDEX_PATH = ROOT / "index" / "faiss_skyline_v1"

def load_docs(data_dir):
    docs = []
    for md_path in sorted(data_dir.glob("*.md")):
        text = md_path.read_text(encoding="utf-8")
        title_match = re.search(r"^# (.+)$", text, re.MULTILINE)
        doc_id_match = re.search(r"\*\*Document ID:\*\* (\S+)", text)
        updated_match = re.search(r"\*\*Last updated:\*\* (\S+)", text)
        docs.append(Document(page_content=text, metadata={
            "source": md_path.name,
            "title": title_match.group(1) if title_match else md_path.stem,
            "doc_id": doc_id_match.group(1) if doc_id_match else md_path.stem,
            "last_updated": updated_match.group(1) if updated_match else "unknown",
        }))
    return docs

chunks = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=80,
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""],
).split_documents(load_docs(DOCS_DIR))

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# save_local overwrites the existing .faiss and .pkl files directly — no need to delete first
INDEX_PATH.mkdir(parents=True, exist_ok=True)
vs = FAISS.from_documents(chunks, embeddings)
vs.save_local(str(INDEX_PATH))
print(f"Rebuilt index: {len(chunks)} chunks → {INDEX_PATH}")

C:\Users\thisi\AppData\Local\Temp\ipykernel_11688\1864052092.py:32: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\thisi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Rebuilt index: 51 chunks → c:\Users\thisi\OneDrive\Desktop\Airplane-RAG-Chatbot\index\faiss_skyline_v1


In [3]:
import sys
from pathlib import Path

# In a notebook, cwd() is the notebooks/ folder — go one level up to the project root
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

from rag.pipeline import answer_question, _load_pipeline

_load_pipeline()
print("Pipeline ready")

Project root: c:\Users\thisi\OneDrive\Desktop\Airplane-RAG-Chatbot
Pipeline ready


## 1. Test Dataset

10 questions covering every policy document, with hand-written ground truths pulled directly from the markdown files.
Ground truths are what a perfect answer *should* say — RAGAS uses them to score context recall and context precision.

In [4]:
TEST_SET = [
    {
        "question": "What is the carry-on weight limit in economy class?",
        "ground_truth": "Economy passengers may bring one cabin bag up to 7kg and one personal item up to 3kg.",
    },
    {
        "question": "How much does an extra checked bag cost on a short-haul flight?",
        "ground_truth": "An additional checked bag on a short-haul flight costs GBP 60.",
    },
    {
        "question": "Can I change my flight booking and is there a fee?",
        "ground_truth": "Yes, flight bookings can be changed. Fees depend on the fare type; flexible fares allow free changes while saver fares incur a change fee.",
    },
    {
        "question": "What are the online check-in opening times?",
        "ground_truth": "Online check-in opens 24 hours before departure and closes 2 hours before departure for most flights.",
    },
    {
        "question": "What compensation am I entitled to for a 4-hour delay on a long-haul flight?",
        "ground_truth": "For a long-haul flight delayed by 3–4 hours, passengers are entitled to GBP 520 with a 50% reduction, so GBP 260, provided the delay is not caused by extraordinary circumstances.",
    },
    {
        "question": "What wheelchair assistance is available at the airport?",
        "ground_truth": "SkyLine Airways provides wheelchair assistance from check-in through to the aircraft door. Passengers should request assistance at least 48 hours before departure.",
    },
    {
        "question": "Can I bring my cat in the cabin and what is the fee?",
        "ground_truth": "Yes, cats may travel in the cabin if the combined weight of pet and carrier does not exceed 8kg. The fee is GBP 50 on short-haul and GBP 100 on long-haul flights per direction.",
    },
    {
        "question": "How do I earn Skylines Miles and how many do I get per flight?",
        "ground_truth": "Skylines Miles are earned on SkyLine Airways flights and partner purchases. The number of miles earned depends on the fare class and route distance.",
    },
    {
        "question": "How long does a refund to a credit card take?",
        "ground_truth": "Refunds to credit or debit cards are processed within 7–14 working days.",
    },
    {
        "question": "What are the fees for taking golf clubs on a long-haul flight?",
        "ground_truth": "Taking golf equipment on a long-haul flight costs GBP 70 per direction. The golf bag must weigh no more than 23kg and be packed in a hard or padded golf travel case.",
    },
]

print(f"{len(TEST_SET)} test questions loaded")

10 test questions loaded


## 2. Run the Pipeline on Every Question

We call `answer_question()` for each test question and collect:
- `answer` — what the LLM generated
- `contexts` — the raw text of the retrieved chunks (what RAGAS evaluates against)

In [5]:
from rag.pipeline import _vectorstore

questions, answers, contexts, ground_truths = [], [], [], []

for item in TEST_SET:
    result = answer_question(item["question"], k=4)

    # Retrieve chunks again to get raw page_content for RAGAS
    retrieved_docs = _vectorstore.similarity_search(item["question"], k=4)
    chunk_texts = [doc.page_content for doc in retrieved_docs]

    questions.append(item["question"])
    answers.append(result["answer"])
    contexts.append(chunk_texts)
    ground_truths.append(item["ground_truth"])

    print(f"Q: {item['question'][:60]}...")
    print(f"A: {result['answer'][:120]}...\n")

print("All questions answered.")

Q: What is the carry-on weight limit in economy class?...
A: In economy class, the carry-on weight limit is one cabin bag with a maximum weight of 7kg, plus one personal item with a...

Q: How much does an extra checked bag cost on a short-haul flig...
A: An additional bag on a short-haul flight costs GBP 60 [POL-BAG-002]....

Q: Can I change my flight booking and is there a fee?...
A: Yes, you can change your flight booking, but the fee depends on the fare type:

- **Saver:** Changes not permitted; you ...

Q: What are the online check-in opening times?...
A: Online check-in opens 48 hours before scheduled departure for all flights and closes:

- **Short-haul:** 60 minutes befo...

Q: What compensation am I entitled to for a 4-hour delay on a l...
A: For a 4-hour delay on a long-haul flight, you may be entitled to compensation of GBP 520, but this amount is reduced by ...

Q: What wheelchair assistance is available at the airport?...
A: Wheelchair assistance is provided free of charge

## 3. Build the RAGAS Dataset and Evaluate

RAGAS expects a HuggingFace `Dataset` with four columns:
- `question` — the input question
- `answer` — what the LLM said
- `contexts` — list of retrieved chunk texts (one list per question)
- `ground_truth` — the reference answer

Each metric uses a different combination of these columns. Faithfulness and Answer Relevancy only need question + answer + contexts. Context Precision and Recall also need ground_truth.

In [6]:
import nest_asyncio
nest_asyncio.apply()  # lets ragas run its async evaluation inside Jupyter's event loop

from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

samples = [
    SingleTurnSample(
        user_input=q,
        response=a,
        retrieved_contexts=c,
        reference=gt,
    )
    for q, a, c, gt in zip(questions, answers, contexts, ground_truths)
]

eval_dataset = EvaluationDataset(samples=samples)

print("Running RAGAS evaluation (makes LLM calls — ~2 min)...")

result = evaluate(
    dataset=eval_dataset,
    metrics=[
        Faithfulness(),
        AnswerRelevancy(),
        ContextPrecision(),
        ContextRecall(),
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

print("\nOverall scores:")
print(result)

Running RAGAS evaluation (makes LLM calls — ~2 min)...


Evaluating: 100%|██████████| 40/40 [00:43<00:00,  1.08s/it]



Overall scores:
{'faithfulness': 0.9833, 'answer_relevancy': 0.8705, 'context_precision': 0.7583, 'context_recall': 0.9000}


## 4. Per-Question Breakdown

The overall scores hide where the pipeline struggles. Looking at the per-question scores tells you *which* questions failed and gives you a starting point for improvement.

In [7]:
import pandas as pd

df = result.to_pandas()

metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

# Shorten questions for display
df.insert(0, "question", [q[:55] + "..." for q in questions])

display(df[["question"] + metric_cols].round(3))

print("\nMean scores:")
print(df[metric_cols].mean().round(3))

,question,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the carry-on weight limit in economy c...,1.000,0.987,1.000,1.0
1,How much does an extra checked bag cost on a s...,1.000,0.890,0.500,1.0
2,Can I change my flight booking and is there a ...,1.000,0.906,0.750,1.0
3,What are the online check-in opening times?...,1.000,0.721,0.000,0.0
4,What compensation am I entitled to for a 4-hou...,0.833,0.959,1.000,1.0
5,What wheelchair assistance is available at the...,1.000,0.854,0.333,1.0
6,Can I bring my cat in the cabin and what is th...,1.000,0.727,1.000,1.0
7,How do I earn Skylines Miles and how many do I...,1.000,0.779,1.000,1.0
8,How long does a refund to a credit card take?...,1.000,0.918,1.000,1.0
9,What are the fees for taking golf clubs on a l...,1.000,0.963,1.000,1.0



Mean scores:
faithfulness         0.983
answer_relevancy     0.870
context_precision    0.758
context_recall       0.900
dtype: float64


## 5. What the Scores Tell Us

**Faithfulness** — if this is below ~0.8, the LLM is making things up that aren't in the retrieved chunks. Fix: tighten the system prompt or reduce `k` so there's less irrelevant context to hallucinate from.

**Answer Relevancy** — if this is low, the LLM is going off-topic. Fix: check the system prompt's instruction clarity.

**Context Precision** — if this is low, we're retrieving chunks that aren't relevant to the question. Fix: reduce `k`, try a better embedding model, or add metadata filtering.

**Context Recall** — if this is low, we're missing chunks that contain the answer. Fix: increase `k`, reduce `chunk_size`, or try hybrid search (keyword + semantic).

**The core trade-off:** Precision and Recall pull in opposite directions on `k`. Higher `k` = better recall but worse precision (and more noise for the LLM). Stage 6 (re-ranking) is the standard fix for this.